# Amazon Reviews — exploratory data analysis

Amazon product reviews labelled with a 1–5 star rating (regression target, scored on balanced MSE). The dataset is used to study temporal drift, so the views below focus on how review volume, the rating distribution, the category mix, and review text shift over time.

In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from drift_happens.dataset.amazon_reviews_23.load import load_amazon_reviews_23

plt.rcParams["axes.unicode_minus"] = False
PRIMARY, SECONDARY = "#4C78A8", "#E45756"

amazon = load_amazon_reviews_23().to_pandas()
amazon["year"] = amazon["timestamp"].dt.year
amazon[["timestamp", "rating", "category", "title"]].head()

## Dataset at a glance

In [ ]:
missing = amazon[["rating", "timestamp", "category"]].isna().mean().max()
pd.Series({
    "Samples": f"{len(amazon):,}",
    "Years": f"{amazon['year'].min()}–{amazon['year'].max()}",
    "Categories": str(amazon["category"].nunique()),
    "Target range": f"{amazon['rating'].min()}–{amazon['rating'].max()} stars",
    "Missing (rating, time, category)": f"{missing:.1%}",
    "Task": "rating regression",
    "Metric": "balanced MSE",
}, name="amazon_reviews_23")

## Temporal coverage

In [ ]:
per_year = amazon["year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(per_year.index, per_year.values, color=PRIMARY)
ax.set_xlabel("Year")
ax.set_ylabel("Reviews")
ax.set_title("Reviews per year")
fig.tight_layout()

## Rating distribution

In [ ]:
rating_counts = amazon["rating"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(rating_counts.index, rating_counts.values, color=PRIMARY)
ax.set_xlabel("Star rating")
ax.set_ylabel("Reviews")
ax.set_title("Rating distribution")
fig.tight_layout()

## Rating drift over time

Mean and early-vs-late distribution of the regression target.

In [ ]:
mean_rating = amazon.groupby("year")["rating"].mean()

low, high = amazon["year"].quantile(0.2), amazon["year"].quantile(0.8)
early = amazon.loc[amazon["year"] <= low, "rating"].value_counts(normalize=True)
late = amazon.loc[amazon["year"] >= high, "rating"].value_counts(normalize=True)
stars = range(1, 6)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(mean_rating.index, mean_rating.values, marker="o", color=PRIMARY)
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Mean rating")
axes[0].set_title("Mean star rating per year")

axes[1].bar([s - 0.2 for s in stars], [early.get(s, 0) for s in stars], width=0.4,
            label=f"up to {int(low)}", color=SECONDARY)
axes[1].bar([s + 0.2 for s in stars], [late.get(s, 0) for s in stars], width=0.4,
            label=f"from {int(high)}", color=PRIMARY)
axes[1].set_xticks(list(stars))
axes[1].set_xlabel("Star rating")
axes[1].set_ylabel("Share")
axes[1].set_title("Rating distribution: early vs late")
axes[1].legend()
fig.tight_layout()

## Categories

In [ ]:
cat_counts = amazon["category"].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
cat_counts.sort_values().plot.barh(ax=ax, color=PRIMARY)
ax.set_xlabel("Reviews")
ax.set_title("Reviews per category")
fig.tight_layout()

## Category mix over time

In [ ]:
cat_year = amazon.groupby(["year", "category"]).size().unstack(fill_value=0)
cat_share = cat_year.div(cat_year.sum(axis=1), axis=0)
order = cat_share.sum().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(cat_share[order].T.values, aspect="auto", cmap="magma")
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=7)
ax.set_xticks(range(len(cat_share.index)))
ax.set_xticklabels(cat_share.index, rotation=90)
ax.set_xlabel("Year")
ax.set_title("Category share over time")
fig.colorbar(im, ax=ax, label="Share of reviews")
fig.tight_layout()

## Review length

In [ ]:
words = amazon["text"].str.split().str.len()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(words.clip(upper=300), bins=50, color=PRIMARY)
ax.set_xlabel("Words in review")
ax.set_ylabel("Reviews")
ax.set_title("Review length")
fig.tight_layout()

## Vocabulary drift

Document frequency of each word in the earliest fifth of years versus the latest fifth; the terms that rise track how product language drifts.

In [ ]:
def document_frequency(texts):
    counts = Counter()
    for text in texts:
        counts.update(set(re.findall(r"[a-z]{4,}", text.lower())))
    return counts

low, high = amazon["year"].quantile(0.2), amazon["year"].quantile(0.8)
early = amazon.loc[amazon["year"] <= low, "text"]
late = amazon.loc[amazon["year"] >= high, "text"]
early = early.sample(min(40000, len(early)), random_state=0)
late = late.sample(min(40000, len(late)), random_state=0)

terms = pd.DataFrame({
    "early": pd.Series(document_frequency(early)) / len(early),
    "late": pd.Series(document_frequency(late)) / len(late),
}).fillna(0)
rising = (terms["late"] - terms["early"]).sort_values().tail(15)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(rising.index, rising.values, color=PRIMARY)
ax.set_xlabel("Increase in document frequency (late minus early)")
ax.set_title(f"Rising terms: up to {int(low)} vs from {int(high)}")
fig.tight_layout()

## Sample records

In [ ]:
from IPython.display import Markdown, display

records = []
for row in amazon.sample(4, random_state=0).itertuples():
    text = " ".join(row.text.split())[:300]
    records.append(f"**{row.rating}★ · {row.category}** · {row.year}\n\n> {text}…")
display(Markdown("\n\n---\n\n".join(records)))